#### Define notebook Parameters

Pass through th pl_worker

In [11]:
# Framework-compatible parameters with manual fallbacks.
# When pl_worker calls this notebook, it should pass these values.
import json
import uuid

# Basic notebook runtime values
task_id = 50
task_name = "Load RMDD to silver"
lineage_id = str(uuid.uuid4())
limit_rows_for_debugging = False

# Manual watermark values
prev_wm = "2026-06-01 00:00:00"
curr_wm = "2026-06-30 00:00:00"

# Manual connection fallback
server_name = "hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com"
source_connection_settings = "{}"

# Source config
# source_tables: first table is the main source; extra tables are used for joins
source_settings = json.dumps({
    "source_name": "CAN_RMDD",
    "database_name": "CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e",
    "table_config": [
        {
            "schema_name": "dbo",
            "source_tables": ["Address"],
            "target_table": "silver_rmdd_address",
            "primary_keys": ["address_id"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["Client"],
            "target_table": "silver_rmdd_client",
            "primary_keys": ["client_number", "member_firm_code"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["Person"],
            "target_table": "silver_rmdd_person",
            "primary_keys": ["person_number", "member_firm_code"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["PersonRelationship", "Person", "RelationshipType"],
            "target_table": "silver_rmdd_person_relationship",
            "primary_keys": ["person_number_1", "member_firm_code_1", "person_number_2", "member_firm_code_2"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["PersonRelationship", "Person", "RelationshipType"],
            "target_table": "silver_rmdd_relationship_type",
            "primary_keys": ["person_number_1", "member_firm_code_1", "person_number_2", "member_firm_code_2", "relationship_description"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "source_tables": ["Person", "PersonSystemMap"],
            "target_table": "silver_rmdd_person_system_map",
            "primary_keys": ["person_number", "member_firm_code"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        }
    ]
})

# Target config
target_settings = json.dumps({
    "lakehouse_name": "lh_canada_global_mds",
    "schema_name": "silver_rmdd",
    "load_strategy": "merge-scd1",
    "location_root": "Files/silver/rmdd"
})

StatementMeta(, 0126b81d-be03-4761-b3ef-0515aaf192c6, 17, Finished, Available, Finished, False)

#### Define functions and logging

Used for import functions

In [12]:
%run nb_transformation_shared_functions

StatementMeta(, 0126b81d-be03-4761-b3ef-0515aaf192c6, 22, Finished, Available, Finished, True)

In [13]:
# Set up standard framework logging
setup_logging()
logger = logging.getLogger("LoadTransactionalToBase")
logger.setLevel(logging.INFO)

StatementMeta(, 0126b81d-be03-4761-b3ef-0515aaf192c6, 23, Finished, Available, Finished, False)

#### Define variables or parameters

Can manually run or be through worker

In [14]:
# Parse notebook settings
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings
source_connection_settings = json.loads(source_connection_settings) if isinstance(source_connection_settings, str) else source_connection_settings

# Source configuration
source_name = source_settings.get("source_name")
source_database = source_settings.get("database_name")
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse_name = target_settings.get("lakehouse_name")
target_schema = target_settings.get("schema_name")
target_load_strategy = target_settings.get("load_strategy", "merge")
location_root = target_settings.get("location_root")

# SQL connection configuration
server_name = (
    source_connection_settings.get("server_name")
    or source_connection_settings.get("jdbc_hostname")
    or source_connection_settings.get("server")
    or server_name
)

# Validation output
print(source_name)
print(source_database)
print(target_schema)
display(table_config)

StatementMeta(, 0126b81d-be03-4761-b3ef-0515aaf192c6, 24, Finished, Available, Finished, False)

CAN_RMDD
CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e
silver_rmdd


SynapseWidget(Synapse.DataFrame, 82cddb93-add7-4550-ad72-911db408c734)

#### Build JDBC connection

Connects to source SQL database

In [15]:
# Build JDBC connection to source SQL database
jdbc_url = (
    "jdbc:sqlserver://"
    f"{server_name}:1433;"
    f"database={source_database};"
    "encrypt=true;"
)

connection_properties = {
    "accessToken": mssparkutils.credentials.getToken("https://database.windows.net/"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Print JDBC URL only
print(jdbc_url)

StatementMeta(, 0126b81d-be03-4761-b3ef-0515aaf192c6, 25, Finished, Available, Finished, False)

jdbc:sqlserver://hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com:1433;database=CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e;encrypt=true;


#### Ingest source table

Read and dedup clean source

In [16]:
# Find every unique source table needed by the target mappings
source_tables_to_read = {}

for cfg in table_config:
    source_schema = cfg.get("schema_name")
    source_tables = cfg.get("source_tables", [])

    if not source_tables:
        raise ValueError(f"Missing source_tables for {cfg.get('target_table')}")

    for source_table in source_tables:
        source_view = f"src_{to_snake_case(source_table)}"

        source_tables_to_read[source_view] = {
            "schema_name": source_schema,
            "source_table": source_table
        }

# Read source tables and create src_* temp views
for source_view, source_cfg in source_tables_to_read.items():
    source_schema = source_cfg.get("schema_name")
    source_table = source_cfg.get("source_table")
    full_source_name = f"{source_schema}.{source_table}"

    source_query = f"""
    (
        SELECT *
        FROM {full_source_name}
    ) AS src
    """

    # Read source table from SQL
    df_source = spark.read.jdbc(
        url=jdbc_url,
        table=source_query,
        properties=connection_properties
    )

    # Drop exact duplicates only
    df_source = remove_exact_duplicates(df_source)

    # Create source temp view
    df_source.createOrReplaceTempView(source_view)

    # Preview source data
    print(f"Source preview for {source_view} ({full_source_name})")
    display(df_source.limit(3))

StatementMeta(, 0126b81d-be03-4761-b3ef-0515aaf192c6, 26, Finished, Available, Finished, False)

Source preview for src_address (dbo.Address)


SynapseWidget(Synapse.DataFrame, 55086bc2-11bd-49bf-91a5-9258fa677196)

Source preview for src_client (dbo.Client)


SynapseWidget(Synapse.DataFrame, e3b31f48-cbf3-4f87-821a-012effcfb689)

...

SynapseWidget(Synapse.DataFrame, 40f46ee1-3228-4596-9dcc-612880727024)

Source preview for src_relationship_type (dbo.RelationshipType)


SynapseWidget(Synapse.DataFrame, 640b4876-5adb-4b83-9a7b-6379ea934707)

Source preview for src_person_system_map (dbo.PersonSystemMap)


SynapseWidget(Synapse.DataFrame, 2f64fdb7-6eff-48be-8722-22f8d2678a39)

#### Transform source table

Applies mapping and transformation as needed

In [17]:
# Map source views into target shape
for cfg in table_config:
    # Get source tables 
    source_tables = cfg.get("source_tables", [])
    main_source_table = source_tables[0]
    main_source_view = f"src_{to_snake_case(main_source_table)}"

    # Get target tables
    target_table = cfg.get("target_table")
    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    # Build map views
    map_view = f"map_{target_entity}"

    # Build watermark filter for this target table
    watermark_col = cfg.get("incremental_column")
    is_incremental = cfg.get("is_incremental", 0)

    # Map Address
    if target_entity == "address":

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"s.{watermark_col} > '{prev_wm}' AND s.{watermark_col} <= '{curr_wm}'"
        
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY s.AddressID) AS BIGINT) AS address_key,
            s.AddressID                     AS address_id,
            s.AddressTypeCode               AS address_type_code,
            s.Address1                      AS address_1,
            s.Address2                      AS address_2,
            s.Address3                      AS address_3,
            s.Address4                      AS address_4,
            s.Address5                      AS address_5,
            s.City                          AS city,
            s.CountryID                     AS country_id,
            CAST(NULL AS STRING)            AS country_code_iso2,
            s.State                         AS state,
            s.PostalCode                    AS zip_code,
            s.Address1                      AS street_address_check,
            s.Active                        AS is_active,
            s.LastUpdatedDateGMT            AS last_updated_date_utc_at_source
        FROM {main_source_view} s
        WHERE {watermark_filter}
        """)

    # Map Client
    elif target_entity == "client":

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"s.{watermark_col} > '{prev_wm}' AND s.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY s.ClientID) AS BIGINT) AS client_key,
            s.ClientNumber                  AS client_number,
            s.MemberFirmCode                AS member_firm_code,
            s.ClientID                      AS client_id,
            s.PMSClientID                   AS pms_client_id,
            s.ClientName                    AS client_name,
            s.AlternateClientName           AS alternate_client_name,
            s.ContactName                   AS contact_name,
            s.OpenDateGMT                   AS open_date_utc,
            s.CloseDateGMT                  AS close_date_utc,
            s.ClientDUNS                    AS client_duns,
            s.ClientGUDUNS                  AS client_guduns,
            s.Comments                      AS comments,
            s.PMSEntityType                 AS pms_entity_type,
            s.ClientType                    AS client_type,
            s.IsConfidential                AS is_confidential,
            s.SICCode                       AS sic_code,
            CAST(NULL AS STRING)            AS client_sector_key,
            s.SanctionsChecked_YN           AS is_sanctions_checked,
            s.Active                        AS is_active,
            s.CreatedDateGMT                AS created_date_utc_at_source,
            s.LastUpdatedDateGMT            AS last_updated_date_utc_at_source
        FROM {main_source_view} s
        WHERE {watermark_filter}
        """)

    # Map Person
    elif target_entity == "person":

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"s.{watermark_col} > '{prev_wm}' AND s.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY s.PersonID) AS BIGINT) AS person_key,
            s.SystemEmployeeCode            AS person_number,
            s.MemberFirmCode                AS member_firm_code,
            s.PersonID                      AS person_id,
            s.PersonStatusCode              AS person_status_code,
            s.Gender                        AS gender,
            s.Prefix                        AS prefix,
            s.FirstName                     AS first_name,
            s.FirstNameKnownAs              AS first_name_known_as,
            s.MiddleName                    AS middle_name,
            s.LastName                      AS last_name,
            s.Suffix                        AS suffix,
            s.Initials                      AS initials,
            s.PhotoPath                     AS photo_path,
            s.CommencementDateGMT           AS commencement_date_utc,
            s.DepartureDateGMT              AS departure_date_utc,
            s.LeaveStatus                   AS leave_status,
            s.SubTeamID                     AS team_key,
            s.JobTitle                      AS job_title,
            s.JobTitlePMS                   AS job_title_pms,
            s.PracticeGroup                 AS practice_group_code,
            CAST(NULL AS STRING)            AS practice_group_key,
            CAST(s.YearOfCall AS STRING)    AS year_of_call,
            s.LastName                      AS employee_name_reporting,
            CAST(s.DeleteFlag AS BOOLEAN)   AS is_active,
            s.LastUpdatedDateGMT            AS last_updated_date_utc_at_source
        FROM {main_source_view} s
        WHERE {watermark_filter}
        """)

    # Map PersonRelationship
    elif target_entity == "person_relationship":
        person_relationship_view = f"src_{to_snake_case(source_tables[0])}"
        person_view = f"src_{to_snake_case(source_tables[1])}"
        relationship_type_view = f"src_{to_snake_case(source_tables[2])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"pr.{watermark_col} > '{prev_wm}' AND pr.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY pr.PersonID1, pr.PersonID2, pr.RelationshipTypeID) AS BIGINT) AS person_relationship_key,
            p1.SystemEmployeeCode           AS person_number_1,
            p1.MemberFirmCode               AS member_firm_code_1,
            p2.SystemEmployeeCode           AS person_number_2,
            p2.MemberFirmCode               AS member_firm_code_2,
            rt.RelationshipTypeName         AS relationship_description,
            pr.PrimaryFlag                  AS primary_flag,
            pr.Active                       AS is_active,
            pr.LastUpdatedDateGMT           AS last_updated_date_utc_at_source
        FROM {person_relationship_view} pr
        LEFT JOIN {person_view} p1
            ON pr.PersonID1 = p1.PersonID
        LEFT JOIN {person_view} p2
            ON pr.PersonID2 = p2.PersonID
        LEFT JOIN {relationship_type_view} rt
            ON pr.RelationshipTypeID = rt.RelationshipTypeID
        WHERE {watermark_filter}
        """)

    # Map RelationshipType
    elif target_entity == "relationship_type":
        person_relationship_view = f"src_{to_snake_case(source_tables[0])}"
        person_view = f"src_{to_snake_case(source_tables[1])}"
        relationship_type_view = f"src_{to_snake_case(source_tables[2])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"pr.{watermark_col} > '{prev_wm}' AND pr.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY pr.PersonID1, pr.PersonID2, pr.RelationshipTypeID) AS BIGINT) AS person_relationship_key,
            p1.SystemEmployeeCode           AS person_number_1,
            p1.MemberFirmCode               AS member_firm_code_1,
            p2.SystemEmployeeCode           AS person_number_2,
            p2.MemberFirmCode               AS member_firm_code_2,
            rt.RelationshipTypeName         AS relationship_description,
            pr.PrimaryFlag                  AS primary_flag,
            pr.Active                       AS is_active,
            pr.LastUpdatedDateGMT           AS last_updated_date_utc_at_source
        FROM {person_relationship_view} pr
        LEFT JOIN {person_view} p1
            ON pr.PersonID1 = p1.PersonID
        LEFT JOIN {person_view} p2
            ON pr.PersonID2 = p2.PersonID
        LEFT JOIN {relationship_type_view} rt
            ON pr.RelationshipTypeID = rt.RelationshipTypeID
        WHERE {watermark_filter}
        """)

    # Map PersonSystemMap
    elif target_entity == "person_system_map":
        person_view = f"src_{to_snake_case(source_tables[0])}"
        person_system_map_view = f"src_{to_snake_case(source_tables[1])}"

        # Apply watermark
        watermark_filter = "1 = 1"
        if is_incremental == 1 and watermark_col:
            watermark_filter = f"p.{watermark_col} > '{prev_wm}' AND p.{watermark_col} <= '{curr_wm}'"

        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {map_view} AS
        SELECT DISTINCT
            CAST(row_number() OVER (ORDER BY p.PersonID) AS BIGINT) AS personsystemmap_key,
            LTRIM(RTRIM(p.SystemEmployeeCode))          AS person_number,
            LTRIM(RTRIM(p.MemberFirmCode))              AS member_firm_code,
            LTRIM(RTRIM(psm.GPMSExternalPersonID))      AS timekeeper_id,
            LTRIM(RTRIM(psm.DomainName))                AS domain_name,
            LTRIM(RTRIM(psm.GPMSExternalPersonID))      AS unified_sap_person_external_id,
            LTRIM(RTRIM(psm.WindowsLogin))              AS windows_login,
            LTRIM(RTRIM(psm.DMSSystemID))               AS dms_system_id,
            LTRIM(RTRIM(psm.HumanResourcesSystemID))    AS human_resources_system_id
        FROM {person_view} p
        LEFT JOIN {person_system_map_view} psm
            ON p.PersonID = psm.PersonID
        WHERE {watermark_filter}
        """)

    else:
        raise ValueError(f"No mapping defined for {target_table}")

    # Preview mapped data
    print(f"Mapped preview for {map_view}")
    display(spark.table(map_view).limit(3))

StatementMeta(, 0126b81d-be03-4761-b3ef-0515aaf192c6, 27, Finished, Available, Finished, False)

Mapped preview for map_address


SynapseWidget(Synapse.DataFrame, 8ac5ce4c-0743-4faf-ba9a-36593078f830)

Mapped preview for map_client


SynapseWidget(Synapse.DataFrame, ae3c20f4-714a-444e-9c66-ca8285214b78)

...

SynapseWidget(Synapse.DataFrame, df8a7656-3870-43d9-9b3e-fdcc706cb6b6)

Mapped preview for map_relationship_type


SynapseWidget(Synapse.DataFrame, 6b7109ea-9e2c-41ac-8995-fecf4217ed3d)

Mapped preview for map_person_system_map


SynapseWidget(Synapse.DataFrame, 16981d70-9ea2-4179-8da0-09ff877873a1)

#### Add metadata to source table

Applies metadata and _hashed_pk

In [18]:
# Add metadata and _hashed_pk to mapped views
# This creates final tgt_* views used by validation and merge
for cfg in table_config:
    source_schema = cfg.get("schema_name")
    source_tables = cfg.get("source_tables", [])
    target_table = cfg.get("target_table")
    primary_keys = cfg.get("primary_keys", [])

    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    map_view = f"map_{target_entity}"
    target_view = f"tgt_{target_entity}"
    source_file = ", ".join([f"{source_schema}.{source_table}" for source_table in source_tables])

    # Read mapped temp view
    df_target = spark.table(map_view)

    # Build hashed primary key
    pk_expr_cols = [to_snake_case(col) for col in primary_keys]

    # Create _hashed_pk from primary key
    df_target = df_target.withColumn(
        "_hashed_pk",
        F.md5(F.concat_ws("|", *[F.col(col).cast("string") for col in pk_expr_cols]))
    )

    # Get non-business columns to build hashed row
    row_hash_cols = [
        col for col in df_target.columns
        if col not in [df_target.columns[0], "_hashed_pk"] + pk_expr_cols
    ]

    # Create _hashed_row from non-business columns
    df_target = df_target.withColumn(
        "_hashed_row",
        F.md5(F.concat_ws("|", *[F.coalesce(F.col(col).cast("string"), F.lit("<NULL>")) for col in row_hash_cols]))
    )

    # Add framework metadata
    df_target = add_metadata(
        df=df_target,
        ingest_date=str(date.today()),
        file_path=source_file,
        schema_name=source_name,
        table_name=target_table,
        lineage_id=lineage_id
    )

    # Create final target temp view
    df_target.createOrReplaceTempView(target_view)

    # Preview final data
    print(f"Final preview for {target_view}")
    display(df_target.limit(3))

StatementMeta(, 0126b81d-be03-4761-b3ef-0515aaf192c6, 28, Finished, Available, Finished, False)

Final preview for tgt_address


SynapseWidget(Synapse.DataFrame, bcfa5e90-4eff-4be0-97dc-bee1a7af533e)

Final preview for tgt_client


SynapseWidget(Synapse.DataFrame, 5cf66052-3d48-466e-bd9e-228aa2b63336)

...

SynapseWidget(Synapse.DataFrame, 4bdcbe8b-ae97-44fd-836d-df39ae702df1)

Final preview for tgt_relationship_type


SynapseWidget(Synapse.DataFrame, 1a8cc2d6-aa70-4218-b38e-0c211703b475)

Final preview for tgt_person_system_map


SynapseWidget(Synapse.DataFrame, 989e52c0-85ad-4a91-b255-e18346121c51)

#### Check duplicates

Return duplicates before merge 

In [19]:
# Check duplicate hashed keys before merge
for cfg in table_config:
    target_table = cfg.get("target_table")

    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    target_view = f"tgt_{target_entity}"

    # Validate duplicate primary keys
    print(f"Checking duplicates for {target_view}")
    validate_duplicates(spark.table(target_view), "_hashed_pk", 10)

    # to do later
    # DO NOT TRIGGER ERROR IF DUPLICATS, CONTINUE BUT SAVE RECONDS ON dupliate tABLE


StatementMeta(, 0126b81d-be03-4761-b3ef-0515aaf192c6, 29, Finished, Available, Finished, False)

Checking duplicates for tgt_address
Total duplicate _hashed_pk groups: 0


SynapseWidget(Synapse.DataFrame, 2cf543ec-4915-4e62-84fc-5a56ce41d3cd)

Checking duplicates for tgt_client
Total duplicate _hashed_pk groups: 0


SynapseWidget(Synapse.DataFrame, 6a69d9cc-1f11-45dc-86a1-a5dae0cf6217)

...

SynapseWidget(Synapse.DataFrame, 872ce7dd-90b1-4655-97bf-bcb607d31ecf)

Checking duplicates for tgt_relationship_type
Total duplicate _hashed_pk groups: 0


SynapseWidget(Synapse.DataFrame, c5ccfcd1-c06f-4e02-96d5-043d4e17d64f)

Checking duplicates for tgt_person_system_map
Total duplicate _hashed_pk groups: 0


SynapseWidget(Synapse.DataFrame, 7037ece1-6583-4fad-84a3-8760248cc371)

#### Merge to target

Merge to target table based on _hashed_key

In [20]:
# Merge final temp views into existing target silver tables.
# Target table creation is owned by the isolated schema notebook.
for cfg in table_config:
    target_table = cfg.get("target_table")

    target_prefix = f"{target_schema}_"
    target_entity = target_table[len(target_prefix):] if target_table.startswith(target_prefix) else to_snake_case(target_table)

    target_view = f"tgt_{target_entity}"
    target_table_path = f"{location_root}/{target_table}"
    full_target_name = f"{target_schema}.{target_table}"

    # Load delta files.
    df_target = spark.table(target_view)
    load_to_target(df_target, target_table_path, target_load_strategy)

    # Print merge metrics.
    delta_table = DeltaTable.forPath(spark, target_table_path)
    operation_metrics = delta_table.history(1).select("operationMetrics").collect()[0]["operationMetrics"]

    rows_inserted = int(operation_metrics.get("numTargetRowsInserted", operation_metrics.get("numOutputRows", 0)))
    rows_updated = int(operation_metrics.get("numTargetRowsUpdated", 0))
    rows_affected = rows_inserted + rows_updated

    print(full_target_name)
    print(f"rows_affected: {rows_affected}")

StatementMeta(, 0126b81d-be03-4761-b3ef-0515aaf192c6, 30, Finished, Available, Finished, False)

2026-07-01 17:25:58,075 UTC - INFO - Merging new records into Files/silver/rmdd/silver_rmdd_address (LoadTransactionalToBase)
silver_rmdd.silver_rmdd_address
rows_affected: 721
2026-07-01 17:26:04,310 UTC - INFO - Merging new records into Files/silver/rmdd/silver_rmdd_client (LoadTransactionalToBase)
silver_rmdd.silver_rmdd_client
rows_affected: 350
2026-07-01 17:26:09,128 UTC - INFO - Merging new records into Files/silver/rmdd/silver_rmdd_person (LoadTransactionalToBase)
silver_rmdd.silver_rmdd_person
rows_affected: 141
2026-07-01 17:26:13,710 UTC - INFO - Merging new records into Files/silver/rmdd/silver_rmdd_person_relationship (LoadTransactionalToBase)
silver_rmdd.silver_rmdd_person_relationship
rows_affected: 0
2026-07-01 17:26:15,244 UTC - INFO - Merging new records into Files/silver/rmdd/silver_rmdd_relationship_type (LoadTransactionalToBase)
silver_rmdd.silver_rmdd_relationship_type
rows_affected: 0
2026-07-01 17:26:16,509 UTC - INFO - Merging new records into Files/silver/rmdd